### Evidence-Grounded Clinical Argumentation: Reproducible Experiment Notebook
#### Student ID: 202531216

This notebook runs my four-condition diagnostic comparison from a clean checkout through
to the tables and figures used in the report.

The four conditions are:

| Condition | Retrieved guidelines | Graph paths | Generator and verifier | Deterministic resolution |
|---|---|---|---|---|
| `direct` | No | No | No | No |
| `flat_rag` | Yes | No | No | No |
| `graph_rag` | Yes | Yes | No | No |
| `evidence_grounded_argumentation` | Yes | Yes | Yes | Yes |

Inference does not run inside this notebook. The pipeline is a two-machine design. The
notebook builds a sealed package of source, input and output, uploads it to a private
Hugging Face bucket, and submits a detached job that runs `hf_job/run.py` inside a vLLM
container on an A100. The notebook then downloads the results and analyses them. I keep
this boundary because it is what makes the staged inputs, the runtime source and the
terminology subset hash-verifiable after the run.

Four inputs are needed to run this notebook:

- a Hugging Face token with access to the artifacts bucket
- a local UMLS 2026AA release (`MRCONSO.RRF`, `MRSTY.RRF`)
- the DiReCT corpus derived from MIMIC-IV, under a valid data use agreement
- access to the project repository

None of these are distributed with the notebook. All three data inputs are licensed and
must be supplied by whoever runs it.

Submitting a job costs money. A 5-case validation takes minutes. An 88-case development
comparison takes about 40 minutes on `a100-large`. Every cell that submits is behind an
approval flag that defaults to off.

##### How to read this notebook

Sections 1 to 6 prepare and verify. Section 7 submits. Sections 8 to 11 analyse.
Sections 12 and 13 cover work I have specified but cannot yet run, and say why.

Cells that spend money are marked **SUBMITTING** and guard themselves with
`APPROVE_SUBMISSION`. Every other cell only reads or verifies, so it is safe to run at
any time.

##### Before anything else

The imports below need three packages that may not already be present. This is the only
cell that installs anything before the repository is cloned.

In [ ]:
%pip install --quiet python-dotenv pandas matplotlib

#### 1. Imports and run settings

I import everything here. The only exception is section 2.2, where I import the project's
own modules, because those do not exist until the repository has been cloned.

`load_dotenv()` reads settings from a `.env` file in the working directory. I keep the
Hugging Face token there rather than typing it into the notebook, so it never appears in a
saved output. A `.env.example` is supplied alongside this notebook.

In [ ]:
import hashlib
import json
import math
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

print("python: ", sys.version.split()[0])
print("pandas: ", pd.__version__)
print("started:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

##### 1.1 Paths and identifiers

Everything lives under three directories. `ROOT` holds the run. `INPUT` holds the licensed
data I supply. `OUTPUT` holds the tables and figures the report uses, so every figure in
the report has one traceable origin. The code is checked out into `REPO` beneath `ROOT`.

In [ ]:
ROOT = Path(os.environ.get("ROOT", Path.cwd() / "reproduction")).resolve()
REPO = ROOT / "argumentation_schemes"
INPUT = Path(os.environ.get("INPUT", ROOT / "input"))
OUTPUT = ROOT / "output"

INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

REPO_URL = os.environ.get(
    "REPO_URL",
    "https://github.com/dugerij/argumentation_schemes_for_clinical_decisions.git",
)

HF_BUCKET = os.environ.get("HF_BUCKET", "Dugerij/jobs-artifacts")
HF_IMAGE = os.environ.get("HF_IMAGE", "vllm/vllm-openai:v0.18.1")
HF_FLAVOR = os.environ.get("HF_FLAVOR", "a100-large")

# Master safety switch. Nothing is submitted to Hugging Face while this is False.
APPROVE_SUBMISSION = False

print("root: ", ROOT)
print("code: ", REPO)
print("input: ", INPUT)
print("output: ", OUTPUT)
print("bucket: ", HF_BUCKET)
print("submission: ", "enabled" if APPROVE_SUBMISSION else "disabled")

#### 2. Obtain the source

I clone the repository rather than assume a working directory, so that a run always
starts from a known commit. I record the commit hash and reuse it when labelling results.
A figure produced from one commit should never be attributed to another.

In [ ]:
def run(command, cwd=None, check=True, capture=True):
    """Run a shell command and return (returncode, stdout, stderr)."""
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        shell=isinstance(command, str),
        capture_output=capture,
        text=True,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"command failed ({result.returncode}): {command}\n{result.stderr}"
        )
    return result.returncode, (result.stdout or ""), (result.stderr or "")


ROOT.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    print("repository already present; fetching")
    run(["git", "fetch", "--all", "--quiet"], cwd=REPO)
else:
    print("cloning", REPO_URL)
    run(["git", "clone", "--quiet", REPO_URL, str(REPO)])

_, commit, _ = run(["git", "rev-parse", "HEAD"], cwd=REPO)
_, branch, _ = run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO)
_, dirty, _ = run(["git", "status", "--porcelain"], cwd=REPO)

COMMIT = commit.strip()
print("branch", branch.strip())
print("commit", COMMIT)
print("clean " if not dirty.strip() else "MODIFIED WORKING TREE")

##### 2.1 Install dependencies

I install into the active environment. `requirements.txt` pins exact versions. Those pins
are part of the reproducibility claim, so I print them rather than let them resolve
silently.

In [ ]:
requirements = (REPO / "requirements.txt").read_text(encoding="utf-8")
print(requirements)

run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
     str(REPO / "requirements.txt")])
run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub[cli]"])

# work from inside the repository so its packages import by name
os.chdir(REPO)

print("dependencies installed")
print("working directory:", Path.cwd())

##### 2.2 Import the project modules

These two imports cannot go in the first cell, because the repository does not exist until
the cell above has run. `hf_config` gives me the scope definitions, and `protocol_sha256`
gives me the method hash I record when freezing a run.

In [ ]:
from hf_job import config as hf_config
from clinical_cds.experiment import protocol_sha256
from clinical_cds.direct import load_direct_dataset, select_direct_partition
from graphrag_runtime.queries import build_validation_queries

print("scopes available:", list(hf_config.SCOPES))

#### 3. Licensed inputs

Three inputs are licensed and are never bundled with the code: the UMLS release, the
DiReCT corpus derived from MIMIC-IV, and the Hugging Face token.

I set the paths below to wherever they are held. The cell checks that each exists and
reports its size, but never prints its contents. The token is read from the environment or
from a prompt and is not echoed.

In [ ]:
UMLS_META = Path(os.environ.get("UMLS_META", INPUT / "umls/2026AA/META"))
DIRECT = Path(os.environ.get("DIRECT", INPUT / "direct"))

for label, path, expected in [
    ("UMLS MRCONSO", UMLS_META / "MRCONSO.RRF", True),
    ("UMLS MRSTY", UMLS_META / "MRSTY.RRF", True),
    ("DiReCT corpus", DIRECT, True),
]:
    if path.exists():
        size = path.stat().st_size if path.is_file() else None
        print(f"present  {label:15s} {'' if size is None else f'{size/1e6:8.1f} MB'}")
    else:
        print(f"MISSING  {label:15s} {path}")

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN: loaded from .env")
else:
    print("HF_TOKEN: not set. Add it to .env before submitting a job.")

##### 3.1 Build the local UMLS database

`hf_job/prepare.py` reads a derived SQLite database, not the raw release. Only a compact
study subset is later staged into the job package, being the concepts and aliases actually
referenced by the controlled corpus and the case evidence. The raw release never leaves
this machine.

In [ ]:
UMLS_DB = REPO / "output/cache/umls_local.sqlite3"

if UMLS_DB.exists():
    print("existing database", UMLS_DB, f"{UMLS_DB.stat().st_size/1e6:.1f} MB")
else:
    print("no local UMLS database found at", UMLS_DB)
    print("build it from the release before preparing a package")

os.environ["UMLS_DB"] = "output/cache/umls_local.sqlite3"

#### 4. Determinism

Reproducibility here rests on four things being fixed, and on each one being checkable
after the run rather than simply asserted before it.

- **Decoding.** Greedy decoding, `temperature = 0`, a fixed seed.
- **Request ordering.** Requests are issued sequentially. Concurrent batching on a shared
  vLLM server can reorder KV-cache state and perturb outputs.
- **Model identity.** The generator, critic and judge models are pinned by revision, not
  by a floating tag.
- **Inputs.** Query set, retrieval index and terminology subset are hashed into the
  package manifest.

The next cell reads these settings straight out of the checked-out source, so what is
printed is what the code will actually do rather than what I claim it does.

In [ ]:
DETERMINISM_KEYS = [
    "temperature", "seed", "top_p", "top_k", "greedy",
    "max_tokens", "revision", "model", "sequential",
]

config_files = [
    REPO / "hf_job/run.py",
    REPO / "hf_job/config.py",
    REPO / "clinical_cds/model.py",
]

for path in config_files:
    if not path.exists():
        print("missing", path)
        continue
    print(f"\n--- {path.relative_to(REPO)} ---")
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        stripped = line.strip()
        low = stripped.lower()
        if any(f"{key}" in low for key in ("temperature", "seed =", '"seed"', "revision")):
            if not low.startswith("#"):
                print(f"{number:6d}  {stripped[:110]}")

##### 4.1 Declared determinism contract

These are the values my analysis assumes. Section 8.1 reads the same fields back out of
the completed run's terminal manifest and fails if the executed run disagrees with
anything declared here. A determinism claim that is only stated up front is not evidence.

In [ ]:
DETERMINISM_CONTRACT = {
    "temperature": 0,
    "seed": 17,
    "decoding": "greedy",
    "request_ordering": "sequential",
}

pd.DataFrame(
    [{"setting": k, "required_value": v} for k, v in DETERMINISM_CONTRACT.items()]
)

##### 4.2 Two identical runs did not give identical results

Fixing the seed and the temperature does not make this pipeline bit-identical across runs.
Two nominally identical 88-case runs of the *unmodified* baselines have differed by up to
one case. Floating-point non-associativity in batched GPU kernels, and any variation in
server-side scheduling, are enough to change a single borderline argmax.

**Logic:**

The practical consequence is a noise floor of roughly 1/88, or about 1.1 percentage
points. I should not interpret differences smaller than that as effects. The comparison
cells below print this floor next to every accuracy difference so it stays visible.

In [ ]:
NOISE_FLOOR_CASES = 1
def noise_floor(n_cases):
    """Observed run-to-run variation, expressed as a proportion."""
    return NOISE_FLOOR_CASES / n_cases

for n in (5, 12, 88, 423):
    print(f"n = {n:4d}   noise floor ~ {noise_floor(n):.3%}")

#### 5. Experiment scopes

A scope is a sealed set of cases. Each entry in `hf_job/config.py` pins a query file and
its SHA-256, so a run cannot silently drift onto a different case set.

Two scopes are executable today. Section 12 covers what I would have to build for a
held-out test scope, and why it is not simply a configuration change.

In [ ]:
rows = []
for name, spec in hf_config.SCOPES.items():
    query_path = REPO / Path(spec["queries_relative"])
    present = query_path.is_file()
    rows.append({
        "scope": name,
        "cases": spec["case_count"],
        "timeout": spec["timeout"],
        "run_name": spec["run_name"],
        "query_file_present": present,
        "size_kb": round(query_path.stat().st_size / 1024, 1) if present else None,
    })

scope_table = pd.DataFrame(rows)
scope_table

##### 5.1 Verify the sealed query set

The pinned digest is what guarantees that a result refers to the case set it claims to. The
check is cheap, so I run it before every submission.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(4 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


for name, spec in hf_config.SCOPES.items():
    query_path = REPO / Path(spec["queries_relative"])
    if not query_path.is_file():
        print(f"{name:12s} query file missing")
        continue
    actual = sha256_file(query_path)
    expected = spec["queries_sha256"]
    status = "match" if actual == expected else "MISMATCH"
    print(f"{name:12s} {status:9s} {actual[:16]}...")
    if actual != expected:
        print(f"             expected {expected[:16]}...")

##### 5.2 Select the scope for this run

`RUN_SCOPE` and `RUN_ID` are the two settings that define a run. I never reuse an
identifier. `hf_job/prepare.py` refuses to overwrite an existing staging directory, and
that is what stops two different experiments from sharing an output prefix.

In [ ]:
RUN_SCOPE = "validation"          # "validation" (5 cases) or "development" (88 cases)
RUN_PHASE = "comparison"          # comparison | retrieval | evaluation
RUN_CASE_LIMIT = None             # development only; e.g. 12 for the bounded sample
RUN_SAMPLE_SEED = "bounded-development-v1"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dt%H%M%Sz")
RUN_ID = f"{RUN_SCOPE}-{stamp}"

os.environ["RUN_SCOPE"] = RUN_SCOPE
os.environ["RUN_PHASE"] = RUN_PHASE
os.environ["RUN_ID"] = RUN_ID
os.environ["HF_BUCKET"] = HF_BUCKET
os.environ["HF_IMAGE"] = HF_IMAGE
os.environ.pop("RUN_CASE_LIMIT", None)
os.environ.pop("RUN_SAMPLE_SEED", None)
if RUN_CASE_LIMIT:
    os.environ["RUN_CASE_LIMIT"] = str(RUN_CASE_LIMIT)
    os.environ["RUN_SAMPLE_SEED"] = RUN_SAMPLE_SEED

planned_cases = RUN_CASE_LIMIT or hf_config.SCOPES[RUN_SCOPE]["case_count"]
print("run id ", RUN_ID)
print("scope  ", RUN_SCOPE, f"({planned_cases} cases)")
print("phase  ", RUN_PHASE)
print("timeout", hf_config.SCOPES[RUN_SCOPE]["timeout"])

#### 6. Preflight

I run two checks before uploading anything: the test suite, and the sealed package build.
Both are free. A failure in either is a reason not to submit.

In [ ]:
returncode, stdout, stderr = run(
    [sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=False
)
print(stdout[-3000:])
TESTS_PASSED = returncode == 0
print("\ntest suite:", "passed" if TESTS_PASSED else "FAILED")

##### 6.1 Build the sealed package

`hf_job.prepare` writes three directories under `.hf-runs/<RUN_ID>/`: the executable source
snapshot, the immutable inputs of queries, retrieval index and terminology subset, and an
empty output target. It also records a SHA-256 for every staged file.

In [ ]:
returncode, stdout, stderr = run(
    [sys.executable, "-m", "hf_job.prepare"], cwd=REPO, check=False
)
print(stdout or stderr)
PACKAGE_READY = returncode == 0

STAGING = REPO / ".hf-runs" / RUN_ID
if PACKAGE_READY:
    for part in ("source", "input", "output"):
        target = STAGING / part
        count = sum(1 for p in target.rglob("*") if p.is_file())
        print(f"{part:8s} {count:5d} files")

##### 6.2 Inspect the package manifests

This is worth reading before spending money. `case_count` confirms the intended sample
size, `query_sha256` confirms the case set, and `umls.raw_rrf_included` confirms that the
raw licensed release is not being uploaded.

In [ ]:
source_manifest = json.loads(
    (STAGING / "source/query_runtime_manifest.json").read_text(encoding="utf-8")
)
input_manifest = json.loads(
    (STAGING / "input/query_package_manifest.json").read_text(encoding="utf-8")
)

summary = {
    "run_id": source_manifest["run_id"],
    "scope": source_manifest["scope"],
    "phase": source_manifest["phase"],
    "case_count": input_manifest["case_count"],
    "query_sha256": input_manifest["query_sha256"][:16] + "...",
    "runtime_sha256": source_manifest["runtime_sha256"][:16] + "...",
    "protocol_sha256": source_manifest["protocol_sha256"][:16] + "...",
    "umls_release": input_manifest["umls"]["release"],
    "raw_umls_included": input_manifest["umls"]["raw_rrf_included"],
    "patient_documents_indexed": input_manifest["patient_documents_indexed"],
    "source_files": len(source_manifest["files"]),
}
pd.DataFrame([{"field": k, "value": v} for k, v in summary.items()])

#### 7. Upload and submit — SUBMITTING

The cells in this section spend money. They refuse to act unless `APPROVE_SUBMISSION` is
`True` and the preflight checks above passed.

Cost scales with case count and flavour. An 88-case comparison runs about 40 minutes on
`a100-large`, and a 5-case validation takes a few minutes. There is no automatic retry. An
infrastructure failure stops the run and is recorded, rather than being silently repeated
at cost.

In [ ]:
def submission_allowed():
    problems = []
    if not APPROVE_SUBMISSION:
        problems.append("APPROVE_SUBMISSION is False")
    if not TESTS_PASSED:
        problems.append("test suite did not pass")
    if not PACKAGE_READY:
        problems.append("package was not prepared")
    if problems:
        print("submission blocked:")
        for item in problems:
            print("  -", item)
        return False
    return True


if submission_allowed():
    for part in ("source", "input", "output"):
        print("syncing", part)
        run([
            "hf", "buckets", "sync",
            str(STAGING / part),
            f"hf://buckets/{HF_BUCKET}/{RUN_ID}/{part}",
        ], cwd=REPO)
    print("upload complete")

In [ ]:
JOB_ID = None

if submission_allowed():
    command = [
        "hf", "jobs", "run", "--detach",
        "--name", f"argumentation-{RUN_ID}",
        "--flavor", HF_FLAVOR,
        "--timeout", str(hf_config.SCOPES[RUN_SCOPE]["timeout"]),
        "--secrets", "HF_TOKEN",
        "--env", f"RUN_SCOPE={RUN_SCOPE}",
        "--env", f"RUN_PHASE={RUN_PHASE}",
        "--env", f"RUN_ID={RUN_ID}",
        "--env", "PROJECT_ROOT=/workspace/project",
        "--env", "PYTHONPATH=/workspace/project",
        "--env", "QUERY_PACKAGE_ROOT=/workspace/query",
        "--env", f"RUNTIME_SCRATCH_ROOT=/tmp/argumentation-schemes/{RUN_ID}",
        "--env", "OUTPUT_ROOT=/outputs",
        "--env", "VLLM_IPC_ROOT=/tmp/vli",
        "--volume", f"hf://buckets/{HF_BUCKET}/{RUN_ID}/source:/workspace/project:ro",
        "--volume", f"hf://buckets/{HF_BUCKET}/{RUN_ID}/input:/workspace/query:ro",
        "--volume", f"hf://buckets/{HF_BUCKET}/{RUN_ID}/output:/outputs:rw",
        HF_IMAGE,
        "python3", "/workspace/project/hf_job/run.py",
    ]
    _, stdout, _ = run(command, cwd=REPO)
    print(stdout)
    JOB_ID = stdout.strip().split()[-1]
    print("job id", JOB_ID)

##### 7.1 Monitor

This polls until the job reaches a terminal state. It is safe to interrupt and re-run,
because it reads status rather than changing it.

In [ ]:
def poll_job(job_id, interval=120, limit=200):
    for attempt in range(limit):
        _, stdout, _ = run(["hf", "jobs", "inspect", job_id], check=False)
        state = "unknown"
        for token in ("COMPLETED", "RUNNING", "ERROR", "FAILED", "CANCELLED"):
            if token in stdout.upper():
                state = token
                break
        stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
        print(f"{stamp}  {state}")
        if state in {"COMPLETED", "ERROR", "FAILED", "CANCELLED"}:
            return state
        time.sleep(interval)
    return "timeout"


if JOB_ID:
    final_state = poll_job(JOB_ID)
    print("final state", final_state)

##### 7.2 Download the outputs

I download every completed run immediately. Results that exist only in the bucket cannot
be cited, plotted or checked, and a report cannot rest on them.

In [ ]:
DOWNLOADS = REPO / "output/hf-downloads" / RUN_ID

if JOB_ID:
    run([
        "hf", "buckets", "sync",
        f"hf://buckets/{HF_BUCKET}/{RUN_ID}/output",
        str(DOWNLOADS),
    ], cwd=REPO)
    print("downloaded to", DOWNLOADS)

#### 8. Load a completed run

Everything below works on any downloaded run, not only one submitted in this session. I
point `ANALYSIS_RUN_ID` at whichever run I want to analyse.

A run directory holds the terminal manifest and audits at the top level, and the per-case
artifacts under `comparison-<scope>/`. Those are `predictions.jsonl` with one row per case
per condition, `argument_traces.jsonl` with one row per case holding the full argumentation
trace, and `adjudications.jsonl`.

In [ ]:
ANALYSIS_RUN_ID = RUN_ID          # or the identifier of any downloaded run

def run_root(run_id):
    return REPO / "output/hf-downloads" / run_id / "runs" / run_id


def comparison_root(run_id):
    root = run_root(run_id)
    candidates = sorted(root.glob("comparison-*"))
    if not candidates:
        raise FileNotFoundError(f"no comparison directory under {root}")
    return candidates[0]


def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def available_runs():
    base = REPO / "output/hf-downloads"
    if not base.is_dir():
        return []
    found = []
    for candidate in sorted(base.iterdir()):
        try:
            comparison_root(candidate.name)
        except (FileNotFoundError, OSError):
            continue
        found.append(candidate.name)
    return found


runs = available_runs()
print(f"{len(runs)} downloaded runs with comparison output")
for name in runs[-15:]:
    print("  ", name)

##### 8.1 Verify the run before using it

A run is usable only if it terminated cleanly, its gates passed, and its recorded decoding
settings match the contract I declared in section 4.1. This check is what turns the
determinism claim into evidence.

In [ ]:
def verify_run(run_id):
    manifest_path = run_root(run_id) / "terminal_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    checks = []
    status = str(manifest.get("status", "")).lower()
    checks.append(("terminated cleanly", status in {"completed", "success"}, status))

    errors = manifest.get("error_count")
    checks.append(("zero execution errors", errors == 0, errors))

    gates = manifest.get("all_gates_pass")
    checks.append(("structural gates pass", gates is True, gates))

    blob = json.dumps(manifest).lower()
    checks.append(("temperature 0 recorded", '"temperature": 0' in blob, None))
    checks.append(("seed 17 recorded", '"seed": 17' in blob, None))

    frame = pd.DataFrame(
        [{"check": name, "pass": bool(ok), "observed": obs} for name, ok, obs in checks]
    )
    return manifest, frame


try:
    manifest, verification = verify_run(ANALYSIS_RUN_ID)
    display(verification)
    if not verification["pass"].all():
        print("\nThis run failed one or more checks and should not be reported.")
except FileNotFoundError as error:
    print("no terminal manifest:", error)

#### 9. Results

##### 9.1 Condition-level accuracy

My primary endpoint is family-level diagnostic accuracy. A prediction counts as correct
when it names the same diagnostic family as the reference label.

Two properties of this metric matter for interpretation. It is produced by a pinned
post-hoc judge model that is evaluation-only and has no influence on any diagnosis. And
because it scores at family level, it does not penalise a coarser answer. A condition that
returns a parent label where the reference is a child still scores as correct. That
asymmetry becomes important in section 9.2.

In [ ]:
CONDITIONS = ["direct", "flat_rag", "graph_rag", "evidence_grounded_argumentation"]
CONDITION_LABELS = {
    "direct": "Direct",
    "flat_rag": "Flat RAG",
    "graph_rag": "Graph RAG",
    "evidence_grounded_argumentation": "Argumentation",
}


def load_predictions(run_id):
    rows = read_jsonl(comparison_root(run_id) / "predictions.jsonl")
    return pd.DataFrame(rows)


def load_family_summary(run_id):
    path = run_root(run_id) / "clinical_family_summary.json"
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


predictions = load_predictions(ANALYSIS_RUN_ID)
print("rows      ", len(predictions))
print("cases     ", predictions["case_id"].nunique())
print("conditions", sorted(predictions["mode"].unique()))
print("\ncolumns:")
print(sorted(predictions.columns.tolist()))

I build the accuracy table from the judge summary where one is present, and recompute it
from the per-case judgments otherwise, so the notebook does not depend on a single file
being available.

In [ ]:
def accuracy_table(run_id):
    judgments_path = run_root(run_id) / "clinical_family_judgments.jsonl"
    if not judgments_path.is_file():
        raise FileNotFoundError(f"no family judgments for {run_id}")

    frame = pd.DataFrame(read_jsonl(judgments_path))
    correct_column = next(
        (c for c in ("family_match", "correct", "is_correct", "pass")
         if c in frame.columns),
        None,
    )
    if correct_column is None:
        raise KeyError(f"no verdict column in {sorted(frame.columns)}")

    grouped = frame.groupby("mode")[correct_column].agg(["sum", "count"])
    grouped["accuracy"] = grouped["sum"] / grouped["count"]
    grouped = grouped.rename(columns={"sum": "correct", "count": "n"})
    grouped = grouped.reindex([c for c in CONDITIONS if c in grouped.index])
    grouped.index = [CONDITION_LABELS.get(i, i) for i in grouped.index]
    return grouped[["correct", "n", "accuracy"]]


results = accuracy_table(ANALYSIS_RUN_ID)
results["accuracy"] = results["accuracy"].round(4)
results["noise_floor"] = [round(noise_floor(n), 4) for n in results["n"]]
results

I plot the same table. Every figure goes to `OUTPUT` under a filename that includes the
run identifier, so a figure in the report can always be traced back to the run that
produced it.

In [ ]:
BAR_COLOURS = ["steelblue", "cadetblue", "seagreen", "indianred"]


def save_figure(fig, name):
    path = OUTPUT / f"{name}_{ANALYSIS_RUN_ID}.png"
    fig.savefig(path, bbox_inches="tight")
    print("saved", path.name)
    return path


fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(results.index, results["accuracy"], color=BAR_COLOURS[: len(results)])
floor = results["noise_floor"].iloc[0]
for index, (name, row) in enumerate(results.iterrows()):
    ax.errorbar(index, row["accuracy"], yerr=floor, color="black", capsize=4, fmt="none")
    ax.text(index, row["accuracy"] + floor + 0.01, f"{row['accuracy']:.3f}",
            ha="center", fontsize=8)
ax.set_ylabel("Family-level accuracy")
ax.set_title(f"Diagnostic accuracy by condition ({ANALYSIS_RUN_ID})", fontsize=9)
ax.set_ylim(0, min(1.0, results["accuracy"].max() + 4 * floor + 0.05))
ax.tick_params(axis="x", rotation=15)
save_figure(fig, "accuracy_by_condition")
plt.show()

**Analysis**

The error bars show the observed run-to-run noise floor, not a sampling confidence
interval. They are there as a reminder that differences of this size are not
interpretable, which is a more useful warning here than a standard error would be.

##### 9.2 Resolver actions: where the argumentation condition actually ends up

The accuracy table on its own is misleading for the argumentation condition, because it
does not distinguish a diagnosis the resolver *established* from one it *fell back to*.

The resolver has confident outcomes, `maintain` and `switch`, and fail-closed outcomes,
`protected_incumbent`, `family_fallback` and `abstain`. A run in which every case exits
through a fail-closed path is not doing argumentative work, whatever its accuracy reads.
This is the most important diagnostic in the notebook.

In [ ]:
def extract(record, *path):
    """Read a nested key path, returning None if any level is absent."""
    current = record
    for key in path:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
        if current is None:
            return None
    return current


def first_present(record, *paths):
    """First path that resolves to a non-None value.

    Uses an explicit None test rather than truthiness, so a legitimate
    ``False`` (the verifier declining to attack) is not mistaken for a
    missing field and silently replaced by the next candidate path.
    """
    for path in paths:
        value = extract(record, *path)
        if value is not None:
            return value
    return None


def resolver_actions(run_id):
    traces = read_jsonl(comparison_root(run_id) / "argument_traces.jsonl")
    rows = []
    for trace in traces:
        action = first_present(
            trace,
            ("resolution_action",),
            ("dialectical_resolution", "action"),
            ("resolver_decision", "action"),
            ("resolver_decision",),
        )
        decision = first_present(
            trace,
            ("direct_differential", "decision"),
            ("dialectical_trace", "direct_differential", "decision"),
        )
        attacked = first_present(
            trace,
            ("attack", "attack"),
            ("dialectical_trace", "attack", "attack"),
        )
        attack_type = first_present(
            trace,
            ("attack", "attack_type"),
            ("dialectical_trace", "attack", "attack_type"),
        )
        rows.append({
            "case_id": trace.get("case_id"),
            "resolution_action": None if action is None else str(action),
            "direct_decision": None if decision is None else str(decision),
            "attacked": None if attacked is None else bool(attacked),
            "attack_type": None if attack_type is None else str(attack_type),
        })
    return pd.DataFrame(rows)


actions = resolver_actions(ANALYSIS_RUN_ID)
action_counts = actions["resolution_action"].value_counts(dropna=False)
print(action_counts.to_string())

def is_confident(action):
    """maintain and switch are the resolver's confident outcomes; the rest fail closed."""
    return str(action).lower() in {"maintain", "switch"}


confident = sum(
    count for name, count in action_counts.items() if is_confident(name)
)
print(f"\nconfident resolutions: {confident} of {len(actions)}")
print(f"fail-closed fallbacks: {len(actions) - confident} of {len(actions)}")

In [ ]:
counts = actions["resolution_action"].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(6, 3.2))
colors = ["seagreen" if is_confident(name) else "indianred" for name in counts.index]
ax.barh([str(i) for i in counts.index], counts.values, color=colors)
for index, value in enumerate(counts.values):
    ax.text(value + 0.4, index, str(value), va="center", fontsize=8)
ax.set_xlabel("Cases")
ax.set_title("Resolver outcome by case (green = confident resolution)", fontsize=9)
ax.invert_yaxis()
save_figure(fig, "resolver_actions")
plt.show()

**Analysis**

Green bars are confident resolutions and red bars are fail-closed exits. If the plot is
entirely red, the resolver never established a diagnosis on its own evidence, and any
accuracy the condition shows is inherited from whatever it fell back to.

##### 9.3 The generator and verifier stages

If confident resolutions are rare, the cause lies upstream. Three quantities localise it:
what the generator decided, whether the verifier attacked at all, and which kind of attack
it raised when it did.

The last of these is decisive. Only a `better_supported_alternative` attack names a
concrete replacement diagnosis, so only that type can produce a `switch`. Every other
validated attack type routes to abstention or family fallback by design. A run can
therefore have plenty of successful attacks and still never switch.

In [ ]:
print("generator decisions")
print(actions["direct_decision"].value_counts(dropna=False).to_string())

print("\nverifier attacked")
print(actions["attacked"].value_counts(dropna=False).to_string())

attacks = actions[actions["attacked"] == True]
if len(attacks):
    print("\nattack types among", len(attacks), "attacks")
    print(attacks["attack_type"].value_counts(dropna=False).to_string())
    switchable = attacks["attack_type"].astype(str).str.contains(
        "better_supported_alternative", case=False
    ).sum()
    print(f"\ncapable of producing a switch: {switchable} of {len(attacks)}")
else:
    print("\nno attacks were raised in this run")

In [ ]:
stages = {
    "Generator supported a diagnosis": int(
        actions["direct_decision"].astype(str)
        .str.contains("diagnosis_supported", case=False).sum()
    ),
    "Verifier raised an attack": int((actions["attacked"] == True).sum()),
    "Attack was better_supported_alternative": int(
        actions["attack_type"].astype(str)
        .str.contains("better_supported_alternative", case=False).sum()
    ),
    "Resolver reached maintain or switch": int(confident),
}

fig, ax = plt.subplots(figsize=(6.4, 3))
ax.barh(list(stages.keys()), list(stages.values()), color="steelblue")
for index, value in enumerate(stages.values()):
    ax.text(value + 0.4, index, str(value), va="center", fontsize=8)
ax.set_xlabel(f"Cases (of {len(actions)})")
ax.set_title("Attrition from generator support to confident resolution", fontsize=9)
ax.invert_yaxis()
save_figure(fig, "resolution_attrition")
plt.show()

**Analysis**

This is the clearest single statement of the mechanism. It shows how many cases survive
each stage of the argumentation pipeline, and therefore exactly where the pipeline loses
them. The gap between any two adjacent bars is the attrition at that stage.

##### 9.4 Does the argumentation condition differ from Graph RAG?

When the resolver falls back to a protected incumbent, it adopts the Graph RAG candidate at
family level. The natural question is how much of the argumentation output is then simply
Graph RAG's answer, possibly generalised to a parent label.

Exact string agreement understates the overlap, because a parent label and its child are
different strings but the same family. I report both.

In [ ]:
def condition_labels(run_id):
    frame = load_predictions(run_id)
    column = next(
        (c for c in ("predicted_label", "prediction", "diagnosis", "label")
         if c in frame.columns),
        None,
    )
    if column is None:
        raise KeyError(f"no prediction column in {sorted(frame.columns)}")
    return frame.pivot_table(
        index="case_id", columns="mode", values=column, aggfunc="first"
    )


labels = condition_labels(ANALYSIS_RUN_ID)
if {"graph_rag", "evidence_grounded_argumentation"} <= set(labels.columns):
    pair = labels[["graph_rag", "evidence_grounded_argumentation"]].dropna()
    identical = (pair["graph_rag"] == pair["evidence_grounded_argumentation"]).sum()
    print(f"identical label:   {identical} of {len(pair)}")
    print(f"different label:   {len(pair) - identical} of {len(pair)}")
    print("\nexamples where they differ:")
    differing = pair[pair["graph_rag"] != pair["evidence_grounded_argumentation"]]
    display(differing.head(12))
else:
    print("both conditions are required for this comparison")

##### 9.5 Paired comparisons

The four conditions run on the same cases, so I compare them pairwise rather than by
reading the accuracy column. Every case is scored under all four conditions, which makes
the comparison paired and lets me use the cases where two conditions disagree.

I use two tests.

- **Exact McNemar** on the discordant cases. It asks whether the split between "only A was
  right" and "only B was right" is further from even than chance would give.
- **Bootstrap confidence interval** on the accuracy difference, resampling cases with a
  fixed seed so the interval is reproducible.

Six pairs means six tests, so I apply a Holm correction. Without it, one apparent result in
six is roughly what I should expect from noise alone.

In [ ]:
def case_correctness(run_id):
    """One row per case, one column per condition, values 1 correct / 0 incorrect."""
    frame = pd.DataFrame(
        read_jsonl(run_root(run_id) / "clinical_family_judgments.jsonl")
    )
    column = next(
        (c for c in ("family_match", "correct", "is_correct", "pass")
         if c in frame.columns),
        None,
    )
    if column is None:
        raise KeyError(f"no verdict column in {sorted(frame.columns)}")
    table = frame.pivot_table(
        index="case_id", columns="mode", values=column, aggfunc="first"
    )
    return table.dropna().astype(int)


correctness = case_correctness(ANALYSIS_RUN_ID)
print("cases scored under every condition:", len(correctness))
correctness.head()

In [ ]:
def mcnemar_exact(a, b):
    """Two-sided exact McNemar. Returns (a_only, b_only, p)."""
    a_only = int(((a == 1) & (b == 0)).sum())
    b_only = int(((a == 0) & (b == 1)).sum())
    n = a_only + b_only
    if n == 0:
        return a_only, b_only, 1.0
    smaller = min(a_only, b_only)
    tail = sum(math.comb(n, i) for i in range(smaller + 1)) / (2 ** n)
    return a_only, b_only, min(1.0, 2 * tail)


def bootstrap_difference(a, b, samples=2000, seed=17):
    """Percentile interval for accuracy(b) - accuracy(a), resampling cases."""
    a, b = list(a), list(b)
    positions = range(len(a))
    generator = random.Random(seed)
    differences = []
    for _ in range(samples):
        picked = [generator.choice(positions) for _ in positions]
        differences.append(
            sum(b[i] - a[i] for i in picked) / len(picked)
        )
    differences.sort()
    low = differences[int(0.025 * samples)]
    high = differences[int(0.975 * samples)]
    return low, high


def holm(p_values):
    """Holm step-down adjustment, preserving input order."""
    ordered = sorted(range(len(p_values)), key=lambda i: p_values[i])
    adjusted = [0.0] * len(p_values)
    running = 0.0
    for rank, index in enumerate(ordered):
        value = (len(p_values) - rank) * p_values[index]
        running = max(running, min(1.0, value))
        adjusted[index] = running
    return adjusted

In [ ]:
present = [c for c in CONDITIONS if c in correctness.columns]

rows = []
for i, first in enumerate(present):
    for second in present[i + 1:]:
        a, b = correctness[first], correctness[second]
        a_only, b_only, p = mcnemar_exact(a, b)
        low, high = bootstrap_difference(a, b)
        rows.append({
            "comparison": f"{CONDITION_LABELS[first]} vs {CONDITION_LABELS[second]}",
            "only_first_correct": a_only,
            "only_second_correct": b_only,
            "difference": round(b.mean() - a.mean(), 4),
            "ci_low": round(low, 4),
            "ci_high": round(high, 4),
            "p_exact": round(p, 4),
        })

paired = pd.DataFrame(rows)
paired["p_holm"] = [round(v, 4) for v in holm(paired["p_exact"].tolist())]
paired["significant"] = paired["p_holm"] < 0.05
paired["ci_excludes_zero"] = (paired["ci_low"] > 0) | (paired["ci_high"] < 0)
paired

**Analysis**

I read this table before I read the accuracy column, because it is the one that says
whether any ranking is real. Three things to check:

- `significant` false across every row means the conditions are statistically
  indistinguishable at this sample size. That is a finding, not a missing result, and it
  should be reported as one.
- `ci_excludes_zero` should agree with `significant`. Where they disagree, the effect is
  marginal and I describe it as such rather than choosing whichever is more favourable.
- `only_first_correct` and `only_second_correct` are the discordant counts. If both are
  small, the conditions are largely agreeing case by case, and the accuracy difference is
  being carried by a handful of cases.

In [ ]:
overlap = pd.DataFrame(
    [[float((correctness[a] == correctness[b]).mean()) for b in present]
     for a in present],
    index=[CONDITION_LABELS[c] for c in present],
    columns=[CONDITION_LABELS[c] for c in present],
).round(3)

fig, ax = plt.subplots(figsize=(4.6, 3.8))
image = ax.imshow(overlap, cmap="Blues", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(present)), overlap.columns, rotation=45, ha="right")
ax.set_yticks(range(len(present)), overlap.index)
for i in range(len(present)):
    for j in range(len(present)):
        ax.text(j, i, f"{overlap.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="black")
ax.set_title("Case-level agreement between conditions", fontsize=9)
fig.colorbar(image, ax=ax, shrink=0.8, label="Cases scored the same")
save_figure(fig, "condition_agreement")
plt.show()

**Analysis**

This shows how often two conditions are right or wrong on the same case, which is not the
same as having similar accuracy. Two conditions can score identically while disagreeing on
many individual cases. High agreement between the argumentation condition and Graph RAG
would be consistent with the fallback behaviour seen in section 9.2.

##### 9.6 Compute cost per condition

No monetary cost is recorded in the run artifacts, so any cost figure here is derived, not
measured. What is recorded is wall-clock runtime and the job flavour, and the per-stage
call structure is known from the architecture. The two retrieval baselines make one
generation call per case, while the argumentation condition adds generator, verifier and
attack-validation calls on top of the same retrieval.

I set `HOURLY_RATE` to the rate actually charged. The table prints its assumption rather
than burying it, because a cost claim without a visible rate is not checkable.

In [ ]:
HOURLY_RATE = 4.00        # currency units per hour for the chosen flavour
CURRENCY = "USD"

def cost_summary(run_id, hourly_rate=HOURLY_RATE):
    manifest = json.loads(
        (run_root(run_id) / "terminal_manifest.json").read_text(encoding="utf-8")
    )
    started = manifest.get("created_at") or manifest.get("started_at")
    finished = manifest.get("completed_at") or manifest.get("finished_at")
    minutes = None
    if started and finished:
        begin = datetime.fromisoformat(str(started).replace("Z", "+00:00"))
        end = datetime.fromisoformat(str(finished).replace("Z", "+00:00"))
        minutes = (end - begin).total_seconds() / 60

    cases = manifest.get("case_count")
    total = (minutes / 60 * hourly_rate) if minutes else None
    return {
        "run_id": run_id,
        "flavor": HF_FLAVOR,
        "assumed_rate": f"{hourly_rate:.2f} {CURRENCY}/hour",
        "runtime_minutes": round(minutes, 1) if minutes else None,
        "cases": cases,
        "estimated_total": round(total, 2) if total else None,
        "estimated_per_case": round(total / cases, 4) if total and cases else None,
    }


pd.DataFrame([cost_summary(ANALYSIS_RUN_ID)]).T.rename(columns={0: "value"})

The per-condition split below is a structural estimate, not a measurement. All four
conditions execute inside one job, so their costs are not separately billed. The estimate
apportions runtime by the number of model calls each condition requires per case. I treat
it as an order-of-magnitude comparison for the discussion, and I say so in the report.

In [ ]:
CALLS_PER_CASE = {
    "direct": 1,
    "flat_rag": 1,
    "graph_rag": 1,
    "evidence_grounded_argumentation": 3,   # generator, verifier, attack validation
}

summary = cost_summary(ANALYSIS_RUN_ID)
total_calls = sum(CALLS_PER_CASE.values())

rows = []
for condition, calls in CALLS_PER_CASE.items():
    share = calls / total_calls
    total = summary["estimated_total"]
    rows.append({
        "condition": CONDITION_LABELS[condition],
        "model_calls_per_case": calls,
        "share_of_compute": round(share, 3),
        "apportioned_cost": round(total * share, 2) if total else None,
    })

cost_frame = pd.DataFrame(rows)
if ANALYSIS_RUN_ID and not results.empty:
    accuracy_map = dict(zip(results.index, results["accuracy"]))
    cost_frame["accuracy"] = cost_frame["condition"].map(accuracy_map)
cost_frame

#### 10. Comparing two runs

I evaluate a method change by comparing two runs on the same case set. That is how the
effect of a fix is measured: hold the scope fixed, change one thing, compare.

Because the noise floor is around one case, a difference of one or two cases in a baseline
condition, which the change did not touch, is my control. If an untouched baseline moves
as much as the treated condition, the comparison has not shown anything.

In [ ]:
def compare_runs(run_a, run_b):
    frames = []
    for label, run_id in (("before", run_a), ("after", run_b)):
        table = accuracy_table(run_id)[["correct", "n", "accuracy"]]
        table.columns = pd.MultiIndex.from_product([[label], table.columns])
        frames.append(table)
    merged = pd.concat(frames, axis=1)
    merged[("change", "accuracy")] = (
        merged[("after", "accuracy")] - merged[("before", "accuracy")]
    )
    merged[("change", "beyond_noise")] = (
        merged[("change", "accuracy")].abs() > merged[("after", "n")].rdiv(1.0)
    )
    return merged.round(4)


RUN_BEFORE = None      # set to a downloaded run identifier
RUN_AFTER = None       # set to a downloaded run identifier

if RUN_BEFORE and RUN_AFTER:
    display(compare_runs(RUN_BEFORE, RUN_AFTER))
else:
    print("set RUN_BEFORE and RUN_AFTER to two downloaded runs on the same scope")

In [ ]:
def compare_mechanism(run_a, run_b):
    rows = []
    for label, run_id in (("before", run_a), ("after", run_b)):
        frame = resolver_actions(run_id)
        rows.append({
            "run": label,
            "run_id": run_id,
            "cases": len(frame),
            "generator_supported": int(
                frame["direct_decision"].astype(str)
                .str.contains("diagnosis_supported", case=False).sum()
            ),
            "attacks_raised": int((frame["attacked"] == True).sum()),
            "better_supported_alternative": int(
                frame["attack_type"].astype(str)
                .str.contains("better_supported_alternative", case=False).sum()
            ),
            "confident_resolutions": int(
                frame["resolution_action"].astype(str)
                .str.contains("maintain|switch", case=False, regex=True).sum()
            ),
        })
    return pd.DataFrame(rows).set_index("run")


if RUN_BEFORE and RUN_AFTER:
    display(compare_mechanism(RUN_BEFORE, RUN_AFTER))

**Analysis**

Mechanism counts are more informative than accuracy for a change of this kind. An
intervention can move the internal behaviour of the pipeline substantially while leaving
the headline accuracy inside the noise floor. That is a real finding rather than a null
one, provided I report it as a mechanism result and not as a performance result.

#### 11. Export tables and figures for the report

I write every file with the run identifier and the source commit recorded alongside it.
The export manifest is the link between a number in the report and the run that produced
it.

In [ ]:
def export_table(frame, name):
    csv_path = OUTPUT / f"{name}_{ANALYSIS_RUN_ID}.csv"
    frame.to_csv(csv_path)
    print("saved", csv_path.name)
    return csv_path


exports = []
exports.append(export_table(results, "accuracy_by_condition"))
exports.append(export_table(
    actions["resolution_action"].value_counts(dropna=False).to_frame("cases"),
    "resolver_actions",
))
exports.append(export_table(cost_frame.set_index("condition"), "compute_cost"))

export_manifest = {
    "run_id": ANALYSIS_RUN_ID,
    "source_commit": COMMIT,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "scope": RUN_SCOPE,
    "cases": int(results["n"].iloc[0]) if not results.empty else None,
    "files": sorted(p.name for p in OUTPUT.glob(f"*{ANALYSIS_RUN_ID}*")),
}

manifest_path = OUTPUT / f"export_manifest_{ANALYSIS_RUN_ID}.json"
manifest_path.write_text(json.dumps(export_manifest, indent=2) + "\n", encoding="utf-8")
print("\n" + json.dumps(export_manifest, indent=2))

#### 12. Held-out evaluation

My corpus is split deterministically. `select_direct_partition` in `clinical_cds/direct.py`
divides the 511 DiReCT cases with `development_fraction=0.2` and `seed=17`, stratified by
diagnosis label. That gives 88 development cases, which the `development` scope already
runs, and 423 held-out test cases.

The split is designed to be development-first and held-out-last. I do not touch the 423
until the method is frozen and the development results have been reviewed, so that nothing
is tuned against the final evaluation set.

**Logic:**

One property of the split is worth stating, because it affects how I read the result. A
diagnosis with fewer than five cases contributes none of them to development, so all of
its cases sit in the held-out partition. The test set therefore contains whole diagnosis
families that development never saw. That makes it a genuine generalisation test rather
than a larger sample of the same thing, and it means I should expect the held-out numbers
to be lower than the development ones rather than treat any drop as a failure.

What is missing is only the wiring. `hf_job/config.py` defines `validation` (5 cases) and
`development` (88), each pinning a query file and its SHA-256. There is no `test` entry, so
there is no `RUN_SCOPE=test` to launch.

Building the held-out query set is cheap. `build_validation_queries` reads the DiReCT
cases and emits, per case, the sectioned patient evidence, a fixed instruction, and the
gold label held aside for evaluation. It does no retrieval and touches no GPU. Retrieval
happens inside the job at run time, not here. So the remaining work is:

- select the `test` partition and build its query records
- write `test_queries.jsonl` and record its SHA-256
- add a `test` entry to `SCOPES` pinning both, with a longer timeout, since 423 cases is
  roughly five times the development runtime
- freeze the method and record the commit, prompt version and protocol hash

The cell below does the first two. It is gated, and it writes only to `OUTPUT`. It does not
modify `SCOPES`, because adding a scope is a deliberate change to the experiment
configuration and belongs in a commit rather than in a notebook run.

In [ ]:
BUILD_HELD_OUT_QUERIES = False

if BUILD_HELD_OUT_QUERIES:
    direct = load_direct_dataset(DIRECT)
    development_cases = select_direct_partition(direct.cases, "development")
    test_cases = select_direct_partition(direct.cases, "test")

    print("total cases:      ", len(direct.cases))
    print("development:      ", len(development_cases))
    print("held-out test:    ", len(test_cases))

    development_ids = {case.case_id for case in development_cases}
    test_ids = {case.case_id for case in test_cases}
    assert not (development_ids & test_ids), "partitions must be disjoint"
    assert len(development_ids | test_ids) == len(direct.cases), "partitions must cover"
    print("partitions are disjoint and cover the pool")
else:
    print("set BUILD_HELD_OUT_QUERIES = True to build the held-out query set")

In [ ]:
if BUILD_HELD_OUT_QUERIES:
    queries = build_validation_queries(direct, [case.case_id for case in test_cases])

    target = OUTPUT / "test_queries.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for record in queries:
            handle.write(json.dumps(record, separators=(",", ":"), sort_keys=True) + "\n")

    digest = sha256_file(target)
    case_ids = sorted(case.case_id for case in test_cases)

    query_manifest = {
        "artifact_id": "msc-v1-test-query-artifact-v1",
        "scope": "test",
        "case_count": len(queries),
        "query_sha256": digest,
        "case_ids_sha256": hashlib.sha256(
            "\n".join(case_ids).encode("utf-8")
        ).hexdigest(),
        "partition_seed": 17,
        "development_fraction": 0.2,
        "previous_model_outputs_used": False,
        "raw_content_in_manifest": False,
        "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    (OUTPUT / "test_query_manifest.json").write_text(
        json.dumps(query_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

    print(json.dumps(query_manifest, indent=2, sort_keys=True))
    print()
    print("Add this to SCOPES in hf_job/config.py, then commit:")
    print(f'    "test": {{')
    print(f'        "queries_relative": Path("<path to test_queries.jsonl>"),')
    print(f'        "queries_sha256": "{digest}",')
    print(f'        "case_count": {len(queries)},')
    print(f'        "timeout": "8h",')
    print(f'        "run_name": "comparison-test",')
    print(f'    }},')

Once that entry is committed, the gate below stops reporting a missing scope and the
held-out run uses the same submission path as every other scope in this notebook.

In [ ]:
HELD_OUT_SCOPE = "test"
APPROVE_HELD_OUT = False

def held_out_ready():
    problems = []
    if HELD_OUT_SCOPE not in hf_config.SCOPES:
        problems.append(f"no {HELD_OUT_SCOPE!r} entry in hf_job/config.py SCOPES")
    else:
        spec = hf_config.SCOPES[HELD_OUT_SCOPE]
        path = REPO / Path(spec["queries_relative"])
        if not path.is_file():
            problems.append(f"sealed query file missing: {path}")
        elif sha256_file(path) != spec["queries_sha256"]:
            problems.append("sealed query digest does not match the pinned value")
    if not APPROVE_HELD_OUT:
        problems.append("APPROVE_HELD_OUT is False")
    return problems


problems = held_out_ready()
if problems:
    print("held-out evaluation is not available:")
    for item in problems:
        print("  -", item)
else:
    print("held-out scope is present and approved")

##### 12.1 Method freeze record

If a held-out run is authorised, this cell records what was frozen, before submission. The
record is what lets me describe the held-out result as held-out.

In [ ]:
def method_freeze_record():
    _, commit, _ = run(["git", "rev-parse", "HEAD"], cwd=REPO)
    _, dirty, _ = run(["git", "status", "--porcelain"], cwd=REPO)
    return {
        "frozen_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "commit": commit.strip(),
        "working_tree_clean": not dirty.strip(),
        "protocol_sha256": protocol_sha256(),
        "determinism_contract": DETERMINISM_CONTRACT,
    }


freeze = method_freeze_record()
print(json.dumps(freeze, indent=2))
if not freeze["working_tree_clean"]:
    print("\nWorking tree is modified. Commit before freezing a method for held-out use.")

#### 13. Out-of-domain stress test: not implemented

An out-of-domain evaluation on MedQA would strengthen my external validity argument,
stratified by whether a question's subject matter is covered by the guideline graphs. The
expected shape of the result is informative either way. If the advantage of the retrieval
and argumentation conditions is concentrated in graph-covered questions, that is direct
evidence that the benefit comes from the graphs rather than from the model.

It is not implemented. There is no MedQA loader, no coverage stratification, and no scope
wiring anywhere in the codebase. Building it means a corpus loader, a mapping from
questions to graph coverage, a query builder producing the same sealed record shape, and a
new scope entry, all outside the currently audited retrieval boundary.

I record it here as specified future work, so the report can describe it as a planned
extension rather than an omission, and so nothing in this notebook implies a result that
was never produced.

#### 14. Findings

I write this after a run completes, from the tables and figures above. Each claim should
name the run identifier and the cell it came from, and should keep apart three kinds of
statement that are easy to blur together:

- what the pipeline produced, being accuracy and counts
- what the pipeline did, being resolver actions and attack types
- what that implies, bounded to this benchmark and this model

The results above are designed to support or refute four points:

- whether the argumentation condition differs from Graph RAG by more than the noise floor,
  and whether any difference reflects independent argumentative work or family-level
  generalisation of the Graph RAG incumbent (sections 9.1, 9.2, 9.4)
- where cases are lost between generator support and confident resolution, and which stage
  is the binding constraint (section 9.3 and the attrition figure)
- whether the schema ordering change moved the mechanism, using the untouched baselines as
  a control for run-to-run variation (section 10)
- what the additional model calls in the argumentation condition cost, and whether the
  auditability they buy is worth that cost on this evidence (section 9.6)

Claims the current evidence does not support, and which must not appear:

- any held-out or generalisation claim, since every result here is development-set
- any clinical utility claim, since this is retrospective benchmark evaluation with no
  clinician validation and no prospective component
- any accuracy difference smaller than the noise floor described as an improvement

#### 15. Reproducibility checklist

| Property | Where it is established |
|---|---|
| Source commit recorded | Section 2 |
| Dependency versions pinned | Section 2.1 |
| Decoding settings fixed and verified | Sections 4, 4.1, 8.1 |
| Run-to-run variation quantified | Section 4.2 |
| Case set sealed and hash-verified | Sections 5.1, 6.2 |
| Licensed data excluded from the package | Section 6.2 |
| Test suite passed before submission | Section 6 |
| Terminal state and gates verified | Section 8.1 |
| Figures traceable to their run | Sections 9, 11 |
| Unavailable analyses labelled | Sections 12, 13 |

Known limitations, which belong in the report and not only here:

- Accuracy is family-level and judged by a model. The judge is pinned and evaluation-only,
  but it is still a model, and its agreement with clinical judgement is not validated here.
- Attack validation claims are taken from the verifier's own report. There is no
  deterministic recomputation of falsification from the cited evidence's polarity.
- All current results are development-set. No held-out estimate exists for this method.
- Per-condition cost is apportioned by call structure, not measured.